## 0. Installation and reproducibility

In [1]:
import platform
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import sklearn
import torch
from sklearn.base import clone
from sklearn.metrics import ndcg_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from scikit_rank import DCNClassifier, DCNRanker

SEED = 42
rng = np.random.default_rng(SEED)
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "sklearn": sklearn.__version__,
    "polars": pl.__version__,
})

{'python': '3.12.13', 'torch': '2.13.0', 'sklearn': '1.9.0', 'polars': '1.42.1'}


## 1. Use sampled MIND-small: a real grouped ranking dataset

Each row is a candidate item. `impression_id` identifies an impression/query and must
not become a predictive feature.

In [2]:
mind_num_features = ["history_len", "entity_embedding_count"]
mind_cat_features = ["user_id", "candidate_news_id", "category", "subcategory"]
MIND_N_TRAIN, MIND_N_TEST = 300, 20  # impressions (groups) sampled per split
MIND_MAX_CANDS = 10                  # optional stage-2 cap per TRAIN impression; None keeps all
MIND_TOP_K = 32                      # keep the TOP_K most frequent cat levels; the rest -> code 0

mind_rng = np.random.default_rng(SEED)

def sample_mind(path: str, n_impressions: int, max_cands: int | None) -> pl.DataFrame:
    lf = pl.scan_parquet(path)
    ids = lf.select("impression_id").unique(maintain_order=True).collect().to_series().to_numpy()
    keep = mind_rng.choice(ids, size=min(n_impressions, len(ids)), replace=False)
    frame = lf.filter(pl.col("impression_id").is_in(keep)).collect()
    if max_cands:
        frame = frame.with_columns(pl.Series("_r", mind_rng.random(frame.height)))
        frame = frame.filter(
            (pl.col("click") == 1) | (pl.col("_r").rank("ordinal").over("impression_id") <= max_cands)
        ).drop("_r")
    return frame.sort("impression_id")

# Processed beforehand following the exps/README.md file.
mind_train = sample_mind("data/mind-small-ranking/processed/temporal/full_train.parquet", MIND_N_TRAIN, MIND_MAX_CANDS)
mind_test = sample_mind("data/mind-small-ranking/processed/temporal/test.parquet", MIND_N_TEST, None)

mind_cat_vocab = {
    c: {v: i + 1 for i, v in enumerate(mind_train[c].drop_nulls().value_counts(sort=True).head(MIND_TOP_K).get_column(c).to_list())}
    for c in mind_cat_features
}

def encode_mind_cats(frame: pl.DataFrame) -> pl.DataFrame:
    return frame.with_columns([
        pl.col(c)
        .cast(pl.String)
        .fill_null("__missing__")
        .replace_strict(list(vocab.keys()), list(vocab.values()), default=0, return_dtype=pl.Int32)
        .alias(c)
        for c, vocab in mind_cat_vocab.items()
    ])

mind_train, mind_test = encode_mind_cats(mind_train), encode_mind_cats(mind_test)
mind_cardinalities = {c: len(v) + 1 for c, v in mind_cat_vocab.items()}  # +1 for code 0

mind_feature_cols = mind_num_features + mind_cat_features
X_train, X_test = mind_train.select(mind_feature_cols).to_pandas(), mind_test.select(mind_feature_cols).to_pandas()
y_train = mind_train["click"].cast(pl.Int64).to_pandas().rename("clicked")
y_test = mind_test["click"].cast(pl.Int64).to_pandas().rename("clicked")
groups_train = mind_train["impression_id"].to_pandas().rename("qid")
groups_test = mind_test["impression_id"].to_pandas().rename("qid")

print(
    f"MIND train: {len(X_train):,} rows / {groups_train.nunique()} impressions\n"
    f"     test: {len(X_test):,} rows / {groups_test.nunique()} impressions"
)
X_train.head().T

MIND train: 2,824 rows / 300 impressions
     test: 806 rows / 20 impressions


,0,1,2,3,4
history_len,21.0,21.0,21.0,21.0,21.0
entity_embedding_count,2.0,5.0,6.0,4.0,1.0
user_id,0.0,0.0,0.0,0.0,0.0
candidate_news_id,0.0,0.0,0.0,0.0,0.0
category,1.0,1.0,5.0,6.0,3.0
subcategory,1.0,6.0,12.0,0.0,0.0


**Helper functions to instantiate DCN models following the Scikit-Learn Interface**

In [3]:
def build_dcnclassifier() -> DCNClassifier:
    return DCNClassifier(
        hidden_units=[32, 16, 8],
        cross_layers=3,
        structure="stacked",
        num_features=mind_num_features,
        cat_features=mind_cat_features,
        num_encoder="plr:n_freq=16;embedding_dim=16",
        cat_encoder="per_feature",
        loss="bce",
        epochs=8,
        batch_size=128,
        lr=3e-3,
        random_state=SEED,
        accelerator_config={"cpu": True},
    )

def build_dcnranker(loss: str = "lambdarank:truncation_level=5") -> DCNRanker:
    return DCNRanker(
        hidden_units=[32, 16, 8],
        cross_layers=2,
        num_features=mind_num_features,
        cat_features=mind_cat_features,
        num_encoder="ple:embedding_dim=8",
        ple_n_bins=16,
        cat_encoder="unified",
        loss=loss,
        epochs=8,
        batch_size=128,
        lr=3e-3,
        random_state=SEED,
        accelerator_config={"cpu": True},
    )

## 2. Replace a gradient-boosting CTR estimator

The surrounding evaluation code is unchanged. Unlike a generic sklearn numeric
estimator, `DCNClassifier` accepts a Pandas frame with string categoricals and
missing numeric values.

In [4]:
from lightgbm import LGBMClassifier
model = LGBMClassifier(random_state=SEED, verbosity=-1).fit(X_train, y_train, categorical_feature=mind_cat_features)

y_score = model.predict_proba(X_test)[:, 1]
print(f"LightGBM test AUC: {roc_auc_score(y_test, y_score):.3f}")

LightGBM test AUC: 0.572


In [5]:
model = build_dcnclassifier().fit(X_train, y_train)

y_score = model.predict_proba(X_test)[:, 1]
print(f"DCN test AUC: {roc_auc_score(y_test, y_score):.3f}")

DCN test AUC: 0.626


## 3. Familiar scikit-learn composition: clone, Pipeline, and CV

All constructor arguments are ordinary estimator parameters, so `clone`,
`set_params`, and search objects can construct an unfitted model.

Cross-validation below is intentionally small to keep a live demo responsive.

In [6]:
pipe = Pipeline(
    [
        ("features", FunctionTransformer(validate=False)),
        ("dcn", build_dcnclassifier()),
    ]
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
cv_auc = cross_val_score(pipe, X_train, y_train, scoring="roc_auc", cv=cv, n_jobs=1)
print(f"3-fold CV AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")

3-fold CV AUC: 0.547 ± 0.018


### Small encoder search

This example searches encoder choices for CTR with AUC, showcasing different configuration options for encoders.

In [7]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    estimator=build_dcnclassifier(),
    param_grid={
        "num_encoder": ["identity", "plr:n_freq=16;embedding_dim=16", "ple:embedding_dim=16"],
        "cat_encoder": ["per_feature", "unified"],
        "dropout": [0.0, 0.2],
    },
    scoring="roc_auc",
    cv=cv,
    n_jobs=1,
    refit=True,
)
search.fit(X_train, y_train)
y_score = search.best_estimator_.predict_proba(X_test)[:, 1]
print("best CV AUC:", round(search.best_score_, 3))
print("best parameters:", search.best_params_)
print(f"DCN test AUC: {roc_auc_score(y_test, y_score):.3f}")

best CV AUC: 0.586
best parameters: {'cat_encoder': 'per_feature', 'dropout': 0.0, 'num_encoder': 'identity'}
DCN test AUC: 0.619


## 4. Train a grouped neural ranker

CTR labels can be used as relevance labels for a minimal ranking example. `DCNRanker` returns uncalibrated ranking scores, not click probabilities.

In [8]:
def mean_ndcg_at_k(y_true, scores, query_ids, k=5):
    """Mean NDCG@k computed independently for each impression/query."""
    frame = pd.DataFrame({"label": np.asarray(y_true), "score": scores, "qid": query_ids})
    return np.mean(
        [
            ndcg_score(
                group[["label"]].to_numpy().reshape(1, -1),
                group[["score"]].to_numpy().reshape(1, -1),
                k=k,
            )
            for _, group in frame.groupby("qid", sort=False)
        ]
    )

ranker = build_dcnranker().fit(X_train, y_train, group=groups_train)

y_score = ranker.predict(X_test)
print(f"NDCG@5: {mean_ndcg_at_k(y_test, y_score, groups_test, k=5):.3f}")

NDCG@5: 0.195


### Switch objectives without rewriting the model

The same `DCNRanker` API accepts pointwise, pairwise, and listwise objectives. Use `group` whenever the chosen loss requires it.

In [9]:
losses_to_try = {
    "pointwise BCE": "bce",
    "pairwise BPR": "bpr",
    "listwise softmax": "listwise",
    "LambdaRank": "lambdarank:truncation_level=5",
}

validation_scores = {}
for name, loss in losses_to_try.items():
    candidate = build_dcnranker(loss=loss).fit(X_train, y_train, group=groups_train)
    validation_scores[name] = mean_ndcg_at_k(y_test, candidate.predict(X_test), groups_test, k=5)

pd.Series(validation_scores, name="NDCG@5").sort_values(ascending=False)

pointwise BCE       0.254696
LambdaRank          0.194570
listwise softmax    0.179914
pairwise BPR        0.167988
Name: NDCG@5, dtype: float64

## 5. Polars eager and streaming LazyFrame inputs

scikit-rank converts eager Pandas, NumPy, and Polars inputs to a common internal
representation. A `LazyFrame` trains through a temporary Arrow IPC stream;
for a lazy input, pass `y` and `group` as column names in the frame. This
avoids eagerly materializing targets and keeps ranking groups intact.

In [10]:
train_pl = pl.from_pandas(X_train).with_columns(
    pl.Series("clicked", y_train.to_numpy()),
    pl.Series("qid", groups_train.to_numpy()),
)
test_pl = pl.from_pandas(X_test).with_columns(pl.Series("qid", groups_test.to_numpy()))

lazy_ranker = build_dcnranker().fit(train_pl.lazy(), y="clicked", group="qid")

# Omit target/group columns at inference, just as with eager data.
lazy_scores = lazy_ranker.predict(test_pl.lazy().drop("qid"))
print(f"NDCG@5: {mean_ndcg_at_k(y_test, y_score, groups_test, k=5):.3f}")

NDCG@5: 0.195


## 6. Include externally computed embeddings

Dense embedding columns are supplied as fixed-width Polars List/Array columns.
`embedding_features` maps an internal stream name to a vector column, and an
embedding encoder can project it before DCN fusion. The example represents
pre-computed user and item vectors.

In [11]:
mind_emb_carry = mind_feature_cols + ["entity_embedding", "click", "impression_id"]
mind_emb_train = mind_train.select(mind_emb_carry).rename({"click": "clicked", "impression_id": "qid"})
mind_emb_test = mind_test.select(mind_emb_carry).rename({"click": "clicked", "impression_id": "qid"})

entity_embedding_ranker = DCNRanker(
    hidden_units=[32, 16, 8],
    cross_layers=2,
    num_features=mind_num_features,
    cat_features=mind_cat_features,
    embedding_features={"entity_embedding": "entity_embedding"},
    embedding_encoders={"entity_embedding": "tower:output_dim=8;dropout=0.0"},
    loss="listwise",
    epochs=3,
    batch_size=128,
    random_state=SEED,
    accelerator_config={"cpu": True},
).fit(mind_emb_train, y="clicked", group="qid")

entity_scores = entity_embedding_ranker.predict(mind_emb_test.drop("clicked", "qid"))
print(entity_scores[:5])
print("embedding input dims:", entity_embedding_ranker.preprocessor_.embedding_input_dims_)
print(f"NDCG@5: {mean_ndcg_at_k(mind_emb_test['clicked'].to_pandas(), entity_scores, mind_emb_test['qid'].to_pandas(), k=5):.3f}")

[-0.03465815  0.01855451  0.0748027   0.04077305  0.01845176]
embedding input dims: {'entity_embedding': 100}
NDCG@5: 0.174


## 7. Train a DCNv2 with a plain PyTorch loop

The estimator wraps preprocessing, batching, and an ignite/Accelerate trainer.
When you want full control - a custom training loop, a research variant, or
integration into an existing PyTorch codebase - drop below the estimator and
compose the same network directly from `scikit_rank.modules`. `DCNv2` is pure
dependency injection: the encoders, cross network, deep tower, reducer, and
head are built separately and handed in. Below we assemble one by hand and
train it with nothing but `torch.optim` and a `DataLoader`.

### Turn the raw table into model-ready tensors

Without the estimator's `preprocessor_` we do the two minimal steps by hand:
standardize numeric columns with train statistics, and map each categorical
to contiguous integer ids with a train-fitted vocabulary (index `0` reserved
for unseen values).

In [12]:
from scikit_rank.modules.dcn import (
    CategoricalEmbeddings,
    CrossNetwork,
    DCNv2,
    DeepNetwork,
    NumericEncoder,
    StackedCrossDeep,
)
from scikit_rank.modules.losses import BCELoss
from scikit_rank.modules.reducers import Concat

torch.manual_seed(SEED)
num_cols, cat_cols = mind_num_features, mind_cat_features

num_mean, num_std = X_train[num_cols].mean(), X_train[num_cols].std().replace(0, 1)
vocab = {c: {v: i + 1 for i, v in enumerate(sorted(X_train[c].unique()))} for c in cat_cols}
cardinalities = [len(vocab[c]) + 1 for c in cat_cols]  # +1 for the unseen/OOV slot at index 0

def to_streams(frame):
    num = ((frame[num_cols] - num_mean) / num_std).fillna(0.0).to_numpy(dtype="float32")
    cat = np.stack(
        [frame[c].map(vocab[c]).fillna(0).to_numpy() for c in cat_cols], axis=1
    ).astype("int64")
    return {"num": torch.from_numpy(num), "cat": torch.from_numpy(cat)}

train_streams, test_streams = to_streams(X_train), to_streams(X_test)
y_train_t = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
print("streams:", {k: tuple(v.shape) for k, v in train_streams.items()}, "cardinalities:", cardinalities)

streams: {'num': (2824, 2), 'cat': (2824, 4)} cardinalities: [34, 34, 15, 34]


### Compose the network from `scikit_rank.modules`

In [13]:
embedding_dim = 8
layers = {
    "num": NumericEncoder(len(num_cols)),  # identity numeric stream
    "cat": CategoricalEmbeddings(cardinalities, [embedding_dim] * len(cat_cols)),
}
reducer = Concat(dim=-1)
layers_out_dim = reducer.compute_output_dim({name: layer.output_dim() for name, layer in layers.items()})

cross = CrossNetwork(layers_out_dim, n_layers=2)              # explicit bounded-degree feature crosses
deep = DeepNetwork(layers_out_dim, hidden_units=[32, 16, 8])  # implicit MLP tower
body = StackedCrossDeep(cross, deep)                          # stacked: cross -> deep
head = torch.nn.Linear(deep.output_dim(), 1)

net = DCNv2(layers=layers, reducer=reducer, body=body, head=head)
loss_fn = BCELoss()
print(net)

DCNv2(
  (_layers): ModuleDict(
    (num): NumericEncoder()
    (cat): CategoricalEmbeddings(
      (_embeddings): ModuleList(
        (0-1): 2 x Embedding(34, 8)
        (2): Embedding(15, 8)
        (3): Embedding(34, 8)
      )
    )
  )
  (_reducer): Concat()
  (_body): StackedCrossDeep(
    (_cross): CrossNetwork(
      (_layers): ModuleList(
        (0-1): 2 x CrossLayer(
          (_V): Linear(in_features=34, out_features=34, bias=False)
          (_U): Identity()
          (_gate_linear): Identity()
        )
      )
    )
    (_deep): DeepNetwork(
      (_network): Sequential(
        (0): Sequential(
          (0): Linear(in_features=34, out_features=32, bias=True)
          (1): Identity()
          (2): ReLU()
          (3): Identity()
        )
        (1): Sequential(
          (0): Linear(in_features=32, out_features=16, bias=True)
          (1): Identity()
          (2): ReLU()
          (3): Identity()
        )
        (2): Sequential(
          (0): Linear(in_feature

### Train with `torch.optim` and a `DataLoader`

In [14]:
optimizer = torch.optim.Adam(net.parameters(), lr=3e-3)
dataset = torch.utils.data.TensorDataset(train_streams["num"], train_streams["cat"], y_train_t)
loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)

for epoch in range(20):
    net.train()
    for num_b, cat_b, y_b in loader:
        optimizer.zero_grad()
        loss = loss_fn(net({"num": num_b, "cat": cat_b}), y_b)
        loss.backward()
        optimizer.step()
    if epoch % 5 == 0:
        net.eval()
        with torch.inference_mode():
            test_auc = roc_auc_score(y_test, torch.sigmoid(net(test_streams)).numpy())
        print(f"epoch {epoch:2d}  train loss {loss.item():.4f}  test AUC {test_auc:.3f}")
with torch.inference_mode():
    test_auc = roc_auc_score(y_test, torch.sigmoid(net(test_streams)).numpy())
print(f"epoch {epoch:2d}  train loss {loss.item():.4f}  test AUC {test_auc:.3f}")

epoch  0  train loss 0.7081  test AUC 0.588
epoch  5  train loss 0.1795  test AUC 0.648
epoch 10  train loss 0.2986  test AUC 0.613
epoch 15  train loss 0.3683  test AUC 0.575
epoch 19  train loss 0.3572  test AUC 0.584
